In [1]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch01_valAcc0.0569.weights.h5')#
# cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
input_filepaths = sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path,training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

I0000 00:00:1770156509.517162  447330 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1770156509.579218  447330 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1770156511.058549  447330 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1770156512.682441  447330 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel bi

Filter mask created with shape: (312, 129)
Initial input shape: (None, 312, 512)
After first Conv2D: (None, 156, 64)
Extracting string 1 from filters 0 to 26
String 1 section shape: (None, 26, 64)
String 1 after first Conv1D: (None, 26, 64)
Extracting string 2 from filters 26 to 52
String 2 section shape: (None, 26, 64)
String 2 after first Conv1D: (None, 26, 64)
Extracting string 3 from filters 52 to 78
String 3 section shape: (None, 26, 64)
String 3 after first Conv1D: (None, 26, 64)
Extracting string 4 from filters 78 to 104
String 4 section shape: (None, 26, 64)
String 4 after first Conv1D: (None, 26, 64)
Extracting string 5 from filters 104 to 130
String 5 section shape: (None, 26, 64)
String 5 after first Conv1D: (None, 26, 64)
Extracting string 6 from filters 130 to 156
String 6 section shape: (None, 26, 64)
String 6 after first Conv1D: (None, 26, 64)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 312, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 312, 256,  │          0 │ input_layer[0][0] │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 312, 32,   │        272 │ reshape[0][0]     │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu         │ (None, 312, 32,   │          0 │ conv2d[0][0]      │
│ (LeakyReLU)         │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_1 (Reshape) │ (None, 312, 512)  │          0 │ leaky_re_lu[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 312, 32)   │     81,952 │ reshape_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 312, 32)   │        128 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_1       │ (None, 312, 32)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d   │ (None, 312, 32)   │          0 │ leaky_re_lu_1[0]… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 312, 64)   │     10,304 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 312, 64)   │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_2       │ (None, 312, 64)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 156, 64)   │          0 │ leaky_re_lu_2[0]… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout1d_1 │ (None, 156, 64)   │          0 │ max_pooling1d[0]… │
│ (SpatialDropout1D)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_2 (Lambda)   │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_4 (Lambda)   │ (None, 26, 64)    │          0 │ spatial_dropout1… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 440,177 (1.68 MB)

...

INFO:tensorflow:Assets written to: /tmp/tmpzcfhpe83/assets


Saved artifact at '/tmp/tmpzcfhpe83'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 312, 256, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 129), dtype=tf.float32, name=None)
Captures:
  123650179140496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179142992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179142224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179143760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179143184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179142608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179142800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179143376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179144144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179145104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  123650179144

/home/gerald/anaconda3/envs/gpu/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1770156520.187227  447330 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1770156520.187266  447330 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1770156520.187806  447330 reader.cc:83] Reading SavedModel from: /tmp/tmpzcfhpe83
I0000 00:00:1770156520.189744  447330 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1770156520.189760  447330 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpzcfhpe83
I0000 00:00:1770156520.229393  447330 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1770156520.240037  447330 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1770156520.493681  447330 loader.cc:220] Running initialization op on S

TFLite model saved as guitarmidi.tflite


I0000 00:00:1770156522.493860  447330 local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1770156522.565690  447330 flatbuffer_export.cc:3715] Skipping runtime version metadata in the model. This will be generated by the exporter.
